In [ ]:
!pip install duckdb

In [ ]:
import pandas as pd

In [ ]:
!pip install -q kagglehub

In [ ]:
import kagglehub

path = kagglehub.dataset_download("arashnic/uplift-modeling")

print("Downloaded to:", path)

100%|██████████| 324M/324M [00:02<00:00, 147MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/datasets/arashnic/uplift-modeling/versions/1


In [ ]:
import os

for root, dirs, files in os.walk(path):
    for file in files:
        print(file)

criteo-uplift-v2.1.csv


In [ ]:
import duckdb
import os

csv_path = os.path.join(path, "criteo-uplift-v2.1.csv")

print("File exists:", os.path.exists(csv_path))

File exists: True


In [ ]:
con = duckdb.connect()

con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_csv_auto('{csv_path}')
""").show()

AttributeError: 'duckdb.duckdb.DuckDBPyConnection' object has no attribute 'show'

In [ ]:
result = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_csv_auto('{csv_path}')
""").fetchdf()

display(result)

,column_name,column_type,null,key,default,extra
0,f0,DOUBLE,YES,None,None,None
1,f1,DOUBLE,YES,None,None,None
2,f2,DOUBLE,YES,None,None,None
3,f3,DOUBLE,YES,None,None,None
4,f4,DOUBLE,YES,None,None,None
5,f5,DOUBLE,YES,None,None,None
6,f6,DOUBLE,YES,None,None,None
7,f7,DOUBLE,YES,None,None,None
8,f8,DOUBLE,YES,None,None,None
9,f9,DOUBLE,YES,None,None,None


In [ ]:
result = con.execute(f"""
    SELECT
        treatment,
        COUNT(*) AS total_customers
    FROM read_csv_auto('{csv_path}')
    GROUP BY treatment
    ORDER BY treatment
""").fetchdf()

display(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,treatment,total_customers
0,0,2096937
1,1,11882655


In [ ]:
result = con.execute(f"""
    SELECT
        treatment,
        COUNT(*) AS total_customers,
        SUM(conversion) AS total_conversions,
        ROUND(AVG(conversion) * 100, 4) AS conversion_rate
    FROM read_csv_auto('{csv_path}')
    GROUP BY treatment
    ORDER BY treatment
""").fetchdf()

display(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,treatment,total_customers,total_conversions,conversion_rate
0,0,2096937,4063.0,0.1938
1,1,11882655,36711.0,0.3089


In [ ]:
result = con.execute(f"""
    SELECT
        treatment,
        COUNT(*) AS total_customers,
        SUM(visit) AS total_visits,
        ROUND(AVG(visit) * 100, 4) AS visit_rate,
        SUM(exposure) AS total_exposures,
        ROUND(AVG(exposure) * 100, 4) AS exposure_rate
    FROM read_csv_auto('{csv_path}')
    GROUP BY treatment
    ORDER BY treatment
""").fetchdf()

display(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,treatment,total_customers,total_visits,visit_rate,total_exposures,exposure_rate
0,0,2096937,80105.0,3.8201,0.0,0.0000
1,1,11882655,576824.0,4.8543,428212.0,3.6037


In [ ]:
result = con.execute(f"""
    SELECT
        'treatment' AS column_name,
        COUNT(DISTINCT treatment) AS unique_values,
        MIN(treatment) AS minimum_value,
        MAX(treatment) AS maximum_value
    FROM read_csv_auto('{csv_path}')

    UNION ALL

    SELECT
        'conversion',
        COUNT(DISTINCT conversion),
        MIN(conversion),
        MAX(conversion)
    FROM read_csv_auto('{csv_path}')

    UNION ALL

    SELECT
        'visit',
        COUNT(DISTINCT visit),
        MIN(visit),
        MAX(visit)
    FROM read_csv_auto('{csv_path}')

    UNION ALL

    SELECT
        'exposure',
        COUNT(DISTINCT exposure),
        MIN(exposure),
        MAX(exposure)
    FROM read_csv_auto('{csv_path}')
""").fetchdf()

display(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,unique_values,minimum_value,maximum_value
0,treatment,2,0,1
1,conversion,2,0,1
2,visit,2,0,1
3,exposure,2,0,1


In [ ]:
result = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE treatment IS NULL) AS missing_treatment,
        COUNT(*) FILTER (WHERE conversion IS NULL) AS missing_conversion,
        COUNT(*) FILTER (WHERE visit IS NULL) AS missing_visit,
        COUNT(*) FILTER (WHERE exposure IS NULL) AS missing_exposure
    FROM read_csv_auto('{csv_path}')
""").fetchdf()

display(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,missing_treatment,missing_conversion,missing_visit,missing_exposure
0,13979592,0,0,0,0


In [ ]:
result = con.execute(f"""
    SELECT
        COUNT(*) FILTER (
            WHERE treatment = 0 AND exposure = 1
        ) AS control_with_exposure,

        COUNT(*) FILTER (
            WHERE treatment = 1 AND exposure = 1
        ) AS treated_with_exposure,

        COUNT(*) FILTER (
            WHERE conversion = 1 AND visit = 0
        ) AS conversions_without_visit

    FROM read_csv_auto('{csv_path}')
""").fetchdf()

display(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,control_with_exposure,treated_with_exposure,conversions_without_visit
0,0,428212,0


In [ ]:
from scipy.stats import norm
import numpy as np

# Customer counts
control_n = 2096937
treatment_n = 11882655

# Conversion counts
control_conversions = 4063
treatment_conversions = 36711

# Conversion rates
p_control = control_conversions / control_n
p_treatment = treatment_conversions / treatment_n

# Pooled conversion rate
pooled_rate = (
    control_conversions + treatment_conversions
) / (control_n + treatment_n)

# Standard error under the null hypothesis
se = np.sqrt(
    pooled_rate * (1 - pooled_rate) *
    (1/control_n + 1/treatment_n)
)

# Z statistic
z_score = (p_treatment - p_control) / se

# Two-sided p-value
p_value = 2 * norm.sf(abs(z_score))

print("Control conversion rate:", round(p_control * 100, 4), "%")
print("Treatment conversion rate:", round(p_treatment * 100, 4), "%")
print("Z-score:", round(z_score, 4))
print("P-value:", p_value)

if p_value < 0.05:
    print("Result: Statistically significant difference")
else:
    print("Result: No statistically significant difference")

Control conversion rate: 0.1938 %
Treatment conversion rate: 0.3089 %
Z-score: 28.5165
P-value: 7.30826328583868e-179
Result: Statistically significant difference


In [ ]:
import numpy as np

# Conversion rates
p_control = control_conversions / control_n
p_treatment = treatment_conversions / treatment_n

# Difference in conversion rates
lift = p_treatment - p_control

# Standard error for confidence interval
se_ci = np.sqrt(
    (p_control * (1 - p_control) / control_n) +
    (p_treatment * (1 - p_treatment) / treatment_n)
)

# 95% confidence interval
lower = lift - 1.96 * se_ci
upper = lift + 1.96 * se_ci

print("Absolute conversion lift:", round(lift * 100, 4), "%")
print("95% CI lower bound:", round(lower * 100, 4), "%")
print("95% CI upper bound:", round(upper * 100, 4), "%")

Absolute conversion lift: 0.1152 %
95% CI lower bound: 0.1085 %
95% CI upper bound: 0.1219 %


In [ ]:
result = con.execute(f"""
    SELECT *
    FROM read_csv_auto('{csv_path}')
    LIMIT 5
""").fetchdf()

display(result)

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
1,12.616365,10.059654,9.002689,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
2,12.616365,10.059654,8.964775,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
3,12.616365,10.059654,9.002801,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0
4,12.616365,10.059654,9.037999,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0


In [ ]:
result = con.execute(f"""
    SELECT
        'f0' AS feature,
        MIN(f0) AS min_value,
        MAX(f0) AS max_value,
        ROUND(AVG(f0), 3) AS average,
        ROUND(STDDEV(f0), 3) AS std_dev
    FROM read_csv_auto('{csv_path}')

    UNION ALL

    SELECT 'f1', MIN(f1), MAX(f1),
           ROUND(AVG(f1), 3), ROUND(STDDEV(f1), 3)
    FROM read_csv_auto('{csv_path}')

    UNION ALL

    SELECT 'f2', MIN(f2), MAX(f2),
           ROUND(AVG(f2), 3), ROUND(STDDEV(f2), 3)
    FROM read_csv_auto('{csv_path}')

    UNION ALL

    SELECT 'f3', MIN(f3), MAX(f3),
           ROUND(AVG(f3), 3), ROUND(STDDEV(f3), 3)
    FROM read_csv_auto('{csv_path}')

    UNION ALL

    SELECT 'f4', MIN(f4), MAX(f4),
           ROUND(AVG(f4), 3), ROUND(STDDEV(f4), 3)
    FROM read_csv_auto('{csv_path}')

    UNION ALL

    SELECT 'f5', MIN(f5), MAX(f5),
           ROUND(AVG(f5), 3), ROUND(STDDEV(f5), 3)
    FROM read_csv_auto('{csv_path}')

    UNION ALL

    SELECT 'f6', MIN(f6), MAX(f6),
           ROUND(AVG(f6), 3), ROUND(STDDEV(f6), 3)
    FROM read_csv_auto('{csv_path}')

    UNION ALL

    SELECT 'f7', MIN(f7), MAX(f7),
           ROUND(AVG(f7), 3), ROUND(STDDEV(f7), 3)
    FROM read_csv_auto('{csv_path}')

    UNION ALL

    SELECT 'f8', MIN(f8), MAX(f8),
           ROUND(AVG(f8), 3), ROUND(STDDEV(f8), 3)
    FROM read_csv_auto('{csv_path}')

    UNION ALL

    SELECT 'f9', MIN(f9), MAX(f9),
           ROUND(AVG(f9), 3), ROUND(STDDEV(f9), 3)
    FROM read_csv_auto('{csv_path}')

    UNION ALL

    SELECT 'f10', MIN(f10), MAX(f10),
           ROUND(AVG(f10), 3), ROUND(STDDEV(f10), 3)
    FROM read_csv_auto('{csv_path}')

    UNION ALL

    SELECT 'f11', MIN(f11), MAX(f11),
           ROUND(AVG(f11), 3), ROUND(STDDEV(f11), 3)
    FROM read_csv_auto('{csv_path}')
""").fetchdf()

display(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,feature,min_value,max_value,average,std_dev
0,f0,12.616365,26.745255,19.620,5.377
1,f1,10.059654,16.344187,10.070,0.105
2,f2,8.214383,9.051962,8.447,0.299
3,f3,-8.398387,4.679882,4.179,1.337
4,f4,10.280525,21.123508,10.339,0.343
5,f5,-9.011892,4.115453,4.029,0.431
6,f6,-31.429784,0.294443,-4.155,4.578
7,f7,4.833815,11.998401,5.102,1.205
8,f8,3.635107,3.971858,3.934,0.057
9,f9,13.190056,75.295017,16.028,7.019


In [ ]:

result = con.execute(f"""
WITH customer_segments AS (
    SELECT
        f0,
        treatment,
        conversion,
        NTILE(5) OVER (ORDER BY f0) AS segment
    FROM read_csv_auto('{csv_path}')
)

SELECT
    segment,
    COUNT(*) AS customers,

    SUM(CASE WHEN treatment = 0 THEN 1 ELSE 0 END)
        AS control_customers,

    SUM(CASE WHEN treatment = 1 THEN 1 ELSE 0 END)
        AS treatment_customers,

    ROUND(
        100.0 * SUM(CASE WHEN treatment = 0 THEN conversion ELSE 0 END)
        / NULLIF(SUM(CASE WHEN treatment = 0 THEN 1 ELSE 0 END), 0),
        4
    ) AS control_conversion_rate,

    ROUND(
        100.0 * SUM(CASE WHEN treatment = 1 THEN conversion ELSE 0 END)
        / NULLIF(SUM(CASE WHEN treatment = 1 THEN 1 ELSE 0 END), 0),
        4
    ) AS treatment_conversion_rate

FROM customer_segments
GROUP BY segment
ORDER BY segment
""").fetchdf()

display(result)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,segment,customers,control_customers,treatment_customers,control_conversion_rate,treatment_conversion_rate
0,1,2795919,464874.0,2331045.0,0.1777,0.2377
1,2,2795919,366213.0,2429706.0,0.5906,0.9295
2,3,2795918,418978.0,2376940.0,0.1580,0.2386
3,4,2795918,422756.0,2373162.0,0.0610,0.0777
4,5,2795918,424116.0,2371802.0,0.0363,0.0450


In [ ]:

result["incremental_lift_pp"] = (
    result["treatment_conversion_rate"]
    - result["control_conversion_rate"]
)

result["relative_lift_pct"] = (
    result["incremental_lift_pp"]
    / result["control_conversion_rate"]
) * 100

display(result[[
    "segment",
    "control_conversion_rate",
    "treatment_conversion_rate",
    "incremental_lift_pp",
    "relative_lift_pct"
]])


,segment,control_conversion_rate,treatment_conversion_rate,incremental_lift_pp,relative_lift_pct
0,1,0.1777,0.2377,0.0600,33.764772
1,2,0.5906,0.9295,0.3389,57.382323
2,3,0.1580,0.2386,0.0806,51.012658
3,4,0.0610,0.0777,0.0167,27.377049
4,5,0.0363,0.0450,0.0087,23.966942


In [ ]:

segment_counts = con.execute(f"""
WITH customer_segments AS (
    SELECT
        treatment,
        conversion,
        NTILE(5) OVER (ORDER BY f0) AS segment
    FROM read_csv_auto('{csv_path}')
)

SELECT
    segment,

    SUM(CASE WHEN treatment = 0 THEN 1 ELSE 0 END)
        AS control_customers,

    SUM(CASE WHEN treatment = 0 THEN conversion ELSE 0 END)
        AS control_conversions,

    SUM(CASE WHEN treatment = 1 THEN 1 ELSE 0 END)
        AS treatment_customers,

    SUM(CASE WHEN treatment = 1 THEN conversion ELSE 0 END)
        AS treatment_conversions

FROM customer_segments
GROUP BY segment
ORDER BY segment
""").fetchdf()

display(segment_counts)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,segment,control_customers,control_conversions,treatment_customers,treatment_conversions
0,1,461759.0,837.0,2334160.0,5552.0
1,2,369328.0,2152.0,2426591.0,22574.0
2,3,418978.0,662.0,2376940.0,5672.0
3,4,422756.0,258.0,2373162.0,1845.0
4,5,424116.0,154.0,2371802.0,1068.0


In [ ]:

from scipy.stats import norm
import numpy as np

test_results = []

for index, row in segment_counts.iterrows():

    control_n = row["control_customers"]
    treatment_n = row["treatment_customers"]

    control_conv = row["control_conversions"]
    treatment_conv = row["treatment_conversions"]

    p_control = control_conv / control_n
    p_treatment = treatment_conv / treatment_n

    # Calculate pooled conversion rate
    pooled_rate = (
        control_conv + treatment_conv
    ) / (control_n + treatment_n)

    # Calculate standard error
    standard_error = np.sqrt(
        pooled_rate * (1 - pooled_rate) *
        (1 / control_n + 1 / treatment_n)
    )

    # Calculate z-score and p-value
    z_score = (p_treatment - p_control) / standard_error
    p_value = 2 * norm.sf(abs(z_score))

    test_results.append({
        "segment": int(row["segment"]),
        "z_score": z_score,
        "p_value": p_value
    })

test_results = pd.DataFrame(test_results)

# Bonferroni-adjusted significance threshold
test_results["significant"] = (
    test_results["p_value"] < 0.01
)

display(test_results)


,segment,z_score,p_value,significant
0,1,7.359239,1.849610e-13,True
1,2,21.019947,4.309178e-98,True
2,3,10.120805,4.467284e-24,True
3,4,3.652512,2.596879e-04,True
4,5,2.501897,1.235300e-02,False


In [ ]:

con.execute(f"""
CREATE OR REPLACE TEMP TABLE segmented_data AS

WITH thresholds AS (
    SELECT
        quantile_cont(f0, 0.20) AS q20,
        quantile_cont(f0, 0.40) AS q40,
        quantile_cont(f0, 0.60) AS q60,
        quantile_cont(f0, 0.80) AS q80
    FROM read_csv_auto('{csv_path}')
)

SELECT
    CASE
        WHEN d.f0 <= t.q20 THEN 1
        WHEN d.f0 <= t.q40 THEN 2
        WHEN d.f0 <= t.q60 THEN 3
        WHEN d.f0 <= t.q80 THEN 4
        ELSE 5
    END AS segment,

    d.treatment,
    d.conversion

FROM read_csv_auto('{csv_path}') AS d
CROSS JOIN thresholds AS t
""")

print("Customer segments created successfully!")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Customer segments created successfully!


In [ ]:

segment_counts = con.execute("""
SELECT
    segment,

    COUNT(*) AS customers,

    SUM(CASE WHEN treatment = 0 THEN 1 ELSE 0 END)
        AS control_customers,

    SUM(CASE WHEN treatment = 0 THEN conversion ELSE 0 END)
        AS control_conversions,

    SUM(CASE WHEN treatment = 1 THEN 1 ELSE 0 END)
        AS treatment_customers,

    SUM(CASE WHEN treatment = 1 THEN conversion ELSE 0 END)
        AS treatment_conversions

FROM segmented_data
GROUP BY segment
ORDER BY segment
""").fetchdf()

display(segment_counts)


,segment,customers,control_customers,control_conversions,treatment_customers,treatment_conversions
0,1,3705545,571784.0,851.0,3133761.0,5859.0
1,2,1886292,259303.0,2138.0,1626989.0,22267.0
2,3,2795921,418979.0,662.0,2376942.0,5672.0
3,4,2795915,422755.0,258.0,2373160.0,1845.0
4,5,2795919,424116.0,154.0,2371803.0,1068.0


In [ ]:

segment_results = segment_counts.copy()

# Calculate conversion rates
segment_results["control_conversion_rate"] = (
    segment_results["control_conversions"]
    / segment_results["control_customers"] * 100
)

segment_results["treatment_conversion_rate"] = (
    segment_results["treatment_conversions"]
    / segment_results["treatment_customers"] * 100
)

# Calculate absolute incremental lift
segment_results["incremental_lift_pp"] = (
    segment_results["treatment_conversion_rate"]
    - segment_results["control_conversion_rate"]
)

# Calculate relative lift
segment_results["relative_lift_pct"] = (
    segment_results["incremental_lift_pp"]
    / segment_results["control_conversion_rate"] * 100
)

display(segment_results[[
    "segment",
    "control_conversion_rate",
    "treatment_conversion_rate",
    "incremental_lift_pp",
    "relative_lift_pct"
]].round(4))


,segment,control_conversion_rate,treatment_conversion_rate,incremental_lift_pp,relative_lift_pct
0,1,0.1488,0.1870,0.0381,25.6204
1,2,0.8245,1.3686,0.5441,65.9881
2,3,0.1580,0.2386,0.0806,51.0261
3,4,0.0610,0.0777,0.0167,27.3909
4,5,0.0363,0.0450,0.0087,24.0100


In [ ]:

from scipy.stats import norm
import numpy as np
import pandas as pd

test_results = []

for _, row in segment_counts.iterrows():

    control_n = row["control_customers"]
    treatment_n = row["treatment_customers"]

    control_conv = row["control_conversions"]
    treatment_conv = row["treatment_conversions"]

    p_control = control_conv / control_n
    p_treatment = treatment_conv / treatment_n

    pooled_rate = (
        control_conv + treatment_conv
    ) / (control_n + treatment_n)

    standard_error = np.sqrt(
        pooled_rate * (1 - pooled_rate) *
        (1 / control_n + 1 / treatment_n)
    )

    z_score = (p_treatment - p_control) / standard_error

    p_value = 2 * norm.sf(abs(z_score))

    test_results.append({
        "segment": int(row["segment"]),
        "z_score": z_score,
        "p_value": p_value
    })

test_results = pd.DataFrame(test_results)

test_results["significant"] = (
    test_results["p_value"] < 0.01
)

display(test_results.round(6))


,segment,z_score,p_value,significant
0,1,6.236834,0.000000,True
1,2,22.769328,0.000000,True
2,3,10.120843,0.000000,True
3,4,3.652489,0.000260,True
4,5,2.501892,0.012353,False


In [ ]:

con.execute(f"""
CREATE OR REPLACE TEMP TABLE segmented_data AS

WITH thresholds AS (
    SELECT
        quantile_cont(f0, 0.20) AS q20,
        quantile_cont(f0, 0.40) AS q40,
        quantile_cont(f0, 0.60) AS q60,
        quantile_cont(f0, 0.80) AS q80
    FROM read_csv_auto('{csv_path}')
)

SELECT
    CASE
        WHEN d.f0 <= t.q20 THEN 1
        WHEN d.f0 <= t.q40 THEN 2
        WHEN d.f0 <= t.q60 THEN 3
        WHEN d.f0 <= t.q80 THEN 4
        ELSE 5
    END AS segment,

    d.treatment,
    d.conversion,
    d.visit

FROM read_csv_auto('{csv_path}') AS d
CROSS JOIN thresholds AS t
""")

print("Updated segmentation table created!")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Updated segmentation table created!


In [ ]:

visit_results = con.execute("""
SELECT
    segment,

    SUM(CASE WHEN treatment = 0 THEN 1 ELSE 0 END)
        AS control_customers,

    SUM(CASE WHEN treatment = 0 THEN visit ELSE 0 END)
        AS control_visits,

    SUM(CASE WHEN treatment = 1 THEN 1 ELSE 0 END)
        AS treatment_customers,

    SUM(CASE WHEN treatment = 1 THEN visit ELSE 0 END)
        AS treatment_visits

FROM segmented_data
GROUP BY segment
ORDER BY segment
""").fetchdf()

visit_results["control_visit_rate"] = (
    visit_results["control_visits"]
    / visit_results["control_customers"] * 100
)

visit_results["treatment_visit_rate"] = (
    visit_results["treatment_visits"]
    / visit_results["treatment_customers"] * 100
)

visit_results["incremental_visit_lift_pp"] = (
    visit_results["treatment_visit_rate"]
    - visit_results["control_visit_rate"]
)

display(visit_results.round(4))


,segment,control_customers,control_visits,treatment_customers,treatment_visits,control_visit_rate,treatment_visit_rate,incremental_visit_lift_pp
0,1,571784.0,29526.0,3133761.0,167136.0,5.1638,5.3334,0.1696
1,2,259303.0,24232.0,1626989.0,234440.0,9.3451,14.4094,5.0644
2,3,418979.0,14443.0,2376942.0,102363.0,3.4472,4.3065,0.8593
3,4,422755.0,6653.0,2373160.0,42219.0,1.5737,1.7790,0.2053
4,5,424116.0,5251.0,2371803.0,30666.0,1.2381,1.2929,0.0548


In [ ]:

from scipy.stats import norm
import numpy as np
import pandas as pd

visit_test_results = []

for _, row in visit_results.iterrows():

    control_n = row["control_customers"]
    treatment_n = row["treatment_customers"]

    control_visits = row["control_visits"]
    treatment_visits = row["treatment_visits"]

    p_control = control_visits / control_n
    p_treatment = treatment_visits / treatment_n

    pooled_rate = (
        control_visits + treatment_visits
    ) / (control_n + treatment_n)

    standard_error = np.sqrt(
        pooled_rate * (1 - pooled_rate) *
        (1 / control_n + 1 / treatment_n)
    )

    z_score = (p_treatment - p_control) / standard_error

    p_value = 2 * norm.sf(abs(z_score))

    visit_test_results.append({
        "segment": int(row["segment"]),
        "z_score": z_score,
        "p_value": p_value
    })

visit_test_results = pd.DataFrame(visit_test_results)

visit_test_results["significant"] = (
    visit_test_results["p_value"] < 0.01
)

display(visit_test_results)


,segment,z_score,p_value,significant
0,1,5.259662,1.443203e-07,True
1,2,69.626775,0.000000e+00,True
2,3,25.632419,6.640755e-145,True
3,4,9.383974,6.353167e-21,True
4,5,2.920809,3.491242e-03,True


In [ ]:

# Combine conversion counts and visit counts
funnel_results = segment_counts.merge(
    visit_results,
    on="segment",
    suffixes=("_conversion", "_visit")
)

# Calculate conversion rates among visitors
funnel_results["control_visitor_conversion_rate"] = (
    funnel_results["control_conversions"]
    / funnel_results["control_visits"] * 100
)

funnel_results["treatment_visitor_conversion_rate"] = (
    funnel_results["treatment_conversions"]
    / funnel_results["treatment_visits"] * 100
)

display(funnel_results[[
    "segment",
    "control_visitor_conversion_rate",
    "treatment_visitor_conversion_rate"
]].round(3))


,segment,control_visitor_conversion_rate,treatment_visitor_conversion_rate
0,1,2.882,3.506
1,2,8.823,9.498
2,3,4.584,5.541
3,4,3.878,4.370
4,5,2.933,3.483


In [ ]:

exposure_results = con.execute(f"""
SELECT
    treatment,
    exposure,
    COUNT(*) AS customers,
    SUM(conversion) AS conversions,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (PARTITION BY treatment),
        4
    ) AS percentage_of_group
FROM read_csv_auto('{csv_path}')
GROUP BY treatment, exposure
ORDER BY treatment, exposure
""").fetchdf()

display(exposure_results)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,treatment,exposure,customers,conversions,percentage_of_group
0,0,0,2096937,4063.0,100.0000
1,1,0,11454443,13680.0,96.3963
2,1,1,428212,23031.0,3.6037


In [ ]:

con.execute(f"""
CREATE OR REPLACE TEMP TABLE segmented_data AS

WITH thresholds AS (
    SELECT
        quantile_cont(f0, 0.20) AS q20,
        quantile_cont(f0, 0.40) AS q40,
        quantile_cont(f0, 0.60) AS q60,
        quantile_cont(f0, 0.80) AS q80
    FROM read_csv_auto('{csv_path}')
)

SELECT
    CASE
        WHEN d.f0 <= t.q20 THEN 1
        WHEN d.f0 <= t.q40 THEN 2
        WHEN d.f0 <= t.q60 THEN 3
        WHEN d.f0 <= t.q80 THEN 4
        ELSE 5
    END AS segment,

    d.treatment,
    d.conversion,
    d.visit,
    d.exposure

FROM read_csv_auto('{csv_path}') AS d
CROSS JOIN thresholds AS t
""")

print("Segmentation table updated with exposure!")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Segmentation table updated with exposure!


In [ ]:

segment_exposure = con.execute("""
SELECT
    segment,

    COUNT(*) FILTER (WHERE treatment = 1)
        AS treatment_customers,

    SUM(exposure) AS exposed_customers,

    ROUND(
        100.0 * SUM(exposure) /
        NULLIF(COUNT(*) FILTER (WHERE treatment = 1), 0),
        4
    ) AS exposure_rate

FROM segmented_data
GROUP BY segment
ORDER BY segment
""").fetchdf()

display(segment_exposure)


,segment,treatment_customers,exposed_customers,exposure_rate
0,1,3133761,21341.0,0.6810
1,2,1626989,310995.0,19.1148
2,3,2376942,66619.0,2.8027
3,4,2373160,18633.0,0.7852
4,5,2371803,10624.0,0.4479


In [ ]:

final_summary = segment_results[[
    "segment",
    "control_conversion_rate",
    "treatment_conversion_rate",
    "incremental_lift_pp",
    "relative_lift_pct"
]].merge(
    visit_results[[
        "segment",
        "control_visit_rate",
        "treatment_visit_rate",
        "incremental_visit_lift_pp"
    ]],
    on="segment"
).merge(
    visit_test_results[[
        "segment",
        "z_score",
        "p_value",
        "significant"
    ]].rename(columns={
        "z_score": "visit_z_score",
        "p_value": "visit_p_value",
        "significant": "visit_significant"
    }),
    on="segment"
).merge(
    test_results[[
        "segment",
        "z_score",
        "p_value",
        "significant"
    ]].rename(columns={
        "z_score": "conversion_z_score",
        "p_value": "conversion_p_value",
        "significant": "conversion_significant"
    }),
    on="segment"
).merge(
    segment_exposure[[
        "segment",
        "exposure_rate"
    ]],
    on="segment"
)

display(final_summary.round(4))


,segment,control_conversion_rate,treatment_conversion_rate,incremental_lift_pp,relative_lift_pct,control_visit_rate,treatment_visit_rate,incremental_visit_lift_pp,visit_z_score,visit_p_value,visit_significant,conversion_z_score,conversion_p_value,conversion_significant,exposure_rate
0,1,0.1488,0.1870,0.0381,25.6204,5.1638,5.3334,0.1696,5.2597,0.0000,True,6.2368,0.0000,True,0.6810
1,2,0.8245,1.3686,0.5441,65.9881,9.3451,14.4094,5.0644,69.6268,0.0000,True,22.7693,0.0000,True,19.1148
2,3,0.1580,0.2386,0.0806,51.0261,3.4472,4.3065,0.8593,25.6324,0.0000,True,10.1208,0.0000,True,2.8027
3,4,0.0610,0.0777,0.0167,27.3909,1.5737,1.7790,0.2053,9.3840,0.0000,True,3.6525,0.0003,True,0.7852
4,5,0.0363,0.0450,0.0087,24.0100,1.2381,1.2929,0.0548,2.9208,0.0035,True,2.5019,0.0124,False,0.4479


In [ ]:

# Export the final summary
final_summary.to_csv(
    "marketing_incrementality_summary.csv",
    index=False
)

print("CSV file created successfully!")


CSV file created successfully!


In [ ]:

from google.colab import files

files.download("marketing_incrementality_summary.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>